# Aprendizaje Automático
# Trabajo Práctico 2

Profesor: Juan Luis Crespo Mariño

Instituto Tecnológico de Costa Rica,

Programa Ciencia de Datos

---

Fecha de entrega: 11 de agosto, hora límite las 6:00 pm.

Medio de entrega: Por medio del TEC-Digital.

Entregables: Un archivo jupyter ( .IPYNB ).

BD utilizada: https://archive.ics.uci.edu/dataset/2/adult


Estudiante:
1. **Jose Pablo Ruiz Myrie**
2. **Nikole Villalobos Lopez**


# Notebook 03 — Modelo de Regresión Logística

En este notebook se entrena y evalúa una Regresión Logística como modelo
baseline para clasificar si el ingreso anual de una persona es superior a $50K.
Se utiliza la selección de variables y las decisiones de preprocesado definidas
en los notebooks anteriores.


In [ ]:
pip install ucimlrepo

In [6]:
# ============================================================
# PREPARACIÓN DEL DATASET
# ============================================================

import pandas as pd
import numpy as np

from ucimlrepo import fetch_ucirepo
from sklearn.impute import SimpleImputer

#Cargamos el dataset
df = pd.read_csv("../data/datos_procesados.csv")

print("Dataset preparado:", df.shape)
df.head()


Dataset preparado: (48842, 17)


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income,capital-gain-log,capital-loss-log
0,22,Private,174043,HS-grad,9,Never-married,Craft-repair,Not-in-family,White,Male,0,0,50,United-States,<=50K,0.000000,0.0
1,24,Private,399449,Bachelors,13,Never-married,Sales,Own-child,White,Female,0,0,40,United-States,<=50K,0.000000,0.0
2,44,Self-emp-inc,79521,Bachelors,13,Married-civ-spouse,Farming-fishing,Husband,White,Male,15024,0,55,United-States,>50K.,9.617471,0.0
3,25,Private,352806,HS-grad,9,Divorced,Other-service,Not-in-family,White,Female,0,0,40,Mexico,<=50K,0.000000,0.0
4,56,Self-emp-not-inc,52822,Some-college,10,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,70,United-States,<=50K.,0.000000,0.0


In [ ]:
# ============================================================
#  Selección de algoritmos y partición de datos
# ============================================================

# Se selecciona Regresión logística debido a que el problema que deseamos
# resolver es determinar si una persona gana más de 50k $ o no. Es decir,
# la variable objetivo es cátegorica binaria. Por lo que tenemos una tarea
# de clasificación. El modelo estima la probabilidad de pertenecer a una de las
# clases. La regresión logística pertenece a la familia de modelos lineales.
# Entre sus ventajas están: rapidez, interpretabilidad y que sus
# coeficientes permiten analizar la dirección e importancia relativa de las
# variables

# Se utiliza una partición 80 % entrenamiento / 20 % prueba donde la mayor parte
# de los datos se utiliza para que el modelo aprenda.

In [ ]:
# ============================================================
# Entrenamiento y ajuste de hiperparámetros
# ============================================================
from sklearn.model_selection import train_test_split

# ------------------------------------------------------------
# 1. Preparar variable objetivo
# ------------------------------------------------------------
df_model = df.copy()

# Unificar etiquetas de income
df_model["income"] = (
    df_model["income"]
    .str.strip()
    .str.rstrip(".")
)

# Convertir la variable objetivo a 0 y 1 donde  0 = <=50K
# y 1 = >50K

df_model["income"] = df_model["income"].map({
    "<=50K": 0,
    ">50K": 1
})

# ------------------------------------------------------------
# 2. Selección de variables
# ------------------------------------------------------------

selected_features_model = [
    "age",
    "workclass",
    "education-num",
    "marital-status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "capital-gain-log",
    "capital-loss-log",
    "hours-per-week",
    "native-country"
]

X_model = df_model[selected_features_model]
y_model = df_model["income"]

# ------------------------------------------------------------
# 3.Separamiento de entrenamiento y prueba
# ------------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X_model,
    y_model,
    test_size=0.20,
    random_state=42,
    stratify=y_model
)

# ------------------------------------------------------------
# 4.Ajustamos preprocesado únicamente al conjunto de entrenamiento
# ------------------------------------------------------------
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import OneHotEncoder

from sklearn.impute import SimpleImputer

# Variables numéricas que utilizan StandardScaler
standard_columns = [
    "age",
    "education-num",
    "capital-gain-log",
    "capital-loss-log"
]

# Variables numéricas que utilizan RobustScaler
robust_columns = [
    "hours-per-week"
]

# Variables categóricas
categorical_columns = [
    "workclass",
    "marital-status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "native-country"
]

#Construimos el preprocesador

preprocessor_model = ColumnTransformer(
    transformers=[
        (
            "standard",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]),
            standard_columns
        ),

        (
            "robust",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", RobustScaler())
            ]),
            robust_columns
        ),

        (
            "categorical",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                (
                    "onehot",
                    OneHotEncoder(
                        drop="first",
                        handle_unknown="ignore"
                    )
                )
            ]),
            categorical_columns
        )
    ]
)

# ------------------------------------------------------------
# 5. Regresión logistica baseline
# ------------------------------------------------------------
from sklearn.linear_model import LogisticRegression

logistic_baseline = Pipeline([
    (
        "preprocessor",
        preprocessor_model
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=1000
        )
    )
])

# Entrenamiento
logistic_baseline.fit(
    X_train,
    y_train
)

# Predicciones
y_pred_baseline = logistic_baseline.predict(X_test)

y_prob_baseline = logistic_baseline.predict_proba(X_test)[:, 1]

# ============================================================
# 6 Evaluación comparativa
# ============================================================

# Realizamos la evaluación de la Regresión Logística mediante las métricas
# Accuracy, Precision, Recall, F1 y AUC-ROC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

baseline_metrics = {
    "Modelo": "Regresión Logística - Baseline",

    "Accuracy": accuracy_score(
        y_test,
        y_pred_baseline
    ),

    "Precision": precision_score(
        y_test,
        y_pred_baseline
    ),

    "Recall": recall_score(
        y_test,
        y_pred_baseline
    ),

    "F1": f1_score(
        y_test,
        y_pred_baseline
    ),

    "AUC-ROC": roc_auc_score(
        y_test,
        y_prob_baseline
    )
}

for metric, value in baseline_metrics.items():
    if metric != "Modelo":
        print(f"{metric}: {value:.4f}")




Accuracy: 0.8430
Precision: 0.7136
Recall: 0.5744
F1: 0.6365
AUC-ROC: 0.9002


In [ ]:
# ============================================================
# Ajuste de hiperparámetros - Regresión Logística
# ============================================================

# Definimos diferentes combinaciones de hiperparámetros para analizar su 
# # efecto sobre el rendimiento del modelo

logistic_configurations = [

    {
        "Modelo": "Regresión Logística con C=0.1",
        "C": 0.1,
        "penalty": "l2",
        "class_weight": None
    },

    {
        "Modelo": "Regresión Logística con C=10",
        "C": 10.0,
        "penalty": "l2",
        "class_weight": None
    },

    {
        "Modelo": "Regresión Logística con L1",
        "C": 1.0,
        "penalty": "l1",
        "class_weight": None
    },

    {
        "Modelo": "Regresión Logística con Balanced",
        "C": 1.0,
        "penalty": "l2",
        "class_weight": "balanced"
    }
]

# Creamos una lista para almacenar los resultados
logistic_results = []

# Incluimos primero los resultados del modelo baseline
logistic_results.append(baseline_metrics)

# Entrenamos cada configuración
for config in logistic_configurations:

    logistic_model = Pipeline([
        (
            "preprocessor",
            preprocessor_model
        ),
        (
            "classifier",
            LogisticRegression(
                C=config["C"],
                penalty=config["penalty"],
                solver="liblinear",
                class_weight=config["class_weight"],
                max_iter=1000
            )
        )
    ])

    # Entrenamiento
    logistic_model.fit(X_train, y_train)

    # Predicciones
    y_pred = logistic_model.predict(X_test)
    y_prob = logistic_model.predict_proba(X_test)[:, 1]

    # Calculamos las métricas
    metrics = {

        "Modelo": config["Modelo"],

        "Accuracy": accuracy_score(
            y_test,
            y_pred
        ),

        "Precision": precision_score(
            y_test,
            y_pred
        ),

        "Recall": recall_score(
            y_test,
            y_pred
        ),

        "F1": f1_score(
            y_test,
            y_pred
        ),

        "AUC-ROC": roc_auc_score(
            y_test,
            y_prob
        )
    }

    logistic_results.append(metrics)


# ------------------------------------------------------------
# 6. Comparación de resultados
# ------------------------------------------------------------

logistic_results_df = pd.DataFrame(logistic_results)

logistic_results_df



c:\Users\Usuario\Documents\Tec\Ciencia de Datos\Aprendizaje Automatico\Proyecto_final\proyecto_final_ml\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\Usuario\Documents\Tec\Ciencia de Datos\Aprendizaje Automatico\Proyecto_final\proyecto_final_ml\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a 

,Modelo,Accuracy,Precision,Recall,F1,AUC-ROC
0,Regresión Logística - Baseline,0.842973,0.713603,0.574423,0.636493,0.900174
1,Regresión Logística con C=0.1,0.842768,0.716056,0.568435,0.633763,0.899847
2,Regresión Logística con C=10,0.842973,0.713603,0.574423,0.636493,0.900223
3,Regresión Logística con L1,0.842870,0.713677,0.573567,0.635997,0.900260
4,Regresión Logística con Balanced,0.804279,0.560068,0.849444,0.675051,0.900040


Análisis Comparativo de los 5 modelos.
Al evañuar las distintas configuraciones de hiperparámetros para la regresión logística donde se modifica la regularización (C), el tipo de penalización (L) y ponderación de clases. Se puede observar como las moficacions de la regularización (C) y el tipo de penalización de L2 a L1 condujo a variaciones mínimas en el desempeño. Esto indica que el modelo tiene una baja sensibilidad a dichos hiperparámetros dentro de los valores que se analizan. Por otro lado, al utilizar la ponderación de las clase "balanced" se produjo un cambio considerable en el clasificador, donde, el recall aumentó de 0.5744 a 0.8494 y el F1 de 0.6365 a 0.6751, aunque la accuracy disminuyó de 0.8430 a 0.8043 y la precision de 0.7136 a 0.5601. Estp indica un compromiso entre la capacidad de detectar individuos con ingresos superiores a $50K y el aumento de falsos positivos. La selección del modelo final depende del objetivo para el cual deseamos identificar si la persona gana más o menos de USD $50k, así, si para la investigación fuera especialmente importante identificar correctamente a las personas que ganan más de $50K Balanced sería el modelo ganador. Esto debido a que, el modelo con Balanced está siendo mucho más agresivo al identificar personas como >$50K. Por eso encuentra muchos más positivos reales, pero también genera más falsos positivos. Por otro lado, si únicamente se busca un buen equilibrio general entre Accuracy y Precision se debería mantener el modelo baseline. Para nuestro caso de estudio se ha seleccionado el modelo baseline como el ganador dentro de las regresiones logísticas

In [32]:
# ============================================================
# Interpretación de variables numéricas y categórica binaria
# ============================================================

selected_interpretation_variables = [
    "standard__age",
    "standard__education-num",
    "standard__capital-gain-log",
    "standard__capital-loss-log",
    "robust__hours-per-week",
    "categorical__sex_Male"
]

selected_coef_df = coef_df[coef_df["Variable"].isin(selected_interpretation_variables)].copy()

selected_coef_df


,Variable,Coeficiente,Odds Ratio
1,standard__education-num,0.728896,2.072791
40,categorical__sex_Male,0.662030,1.938723
2,standard__capital-gain-log,0.514271,1.672419
0,standard__age,0.314683,1.369824
3,standard__capital-loss-log,0.236411,1.266694
4,robust__hours-per-week,0.158297,1.171515


Al analizar los coeficientes y los odds ratios de las variables númericas podemos ver como todos los coeficientes son positivos, por lo que se aoscian con una mayor probabilidad de pertenecer al grupo de >50K, manteniendo constantes las demás variables. Así, education-num es la vairbale númerica con mayor peso positivo en la regresión, su coeficiente de 0.729 y odds ratio de 2.073 nos indican que un aumento de una desviación estándar en el nivel educativo se asocia con unas odds aproximadamente 2.07 veces mayores de superar los $50K lo que concuerda con el comportamiento esperado donde se esperaría que una mayor educación permitan el acceso a puestos de mayor remuneración. En cuanto a capital-gain-log con un coeficiente positivo de 0.514 y un odds ratio de 1.672, nos indica que mayores valores de ganancias de capital están asociados con mayores odds de pertenecer al grupo de ingresos superiores a $50K, En otras palabras, un aumento de una desviación estándar en la variable transformada se asocia con unas odds aproximadamente 67.2% mayores, manteniendo constantes las demás características. En cuanto a age, se presenta una asociación positiva moderada con los ingresos superiores a $50K. Su odds ratio de 1.370 indica que un aumento de una desviación estándar en la edad se asocia con unas odds aproximadamente 37.0% mayores de pertenecer a la categoría de ingresos altos, manteniendo constantes las demás variables. Esto concuerda con el comportamiento esperado sobre la acumulación de experiencia laboral, conocimientos y progresión profesional a lo largo del tiempo Con respecto a capital-loss-log, presenta una asociación positiva con los ingresos superiores a $50K. Aquí, el odds ratio de 1.267 indica unas odds aproximadamente 26.7% mayores ante un incremento de una desviación estándar en la variable transformada Ahora bien, una pérdida de capital podría parecer asociada negativamente con el ingreso, no obstante este resultado podría reflejar que los individuos con mayores ingresos presentan una mayor participación en activos o inversiones y, por tanto, también una mayor posibilidad de registrar pérdidas de capital. En cuanto a hours-per-week, el odds ratio de 1.172 indica que aumentos en las horas trabajadas, están asociados con mayores odds de pertenecer al grupo de ingresos superiores a $50K. Sin embargo, el efecto es menor que el observado para variables como educación o ganancias de capital. Por otro lado, la categoría Male presenta un coeficiente positivo de 0.662 y un odds ratio de 1.939. Así, se tiene que manteniendo constantes las demás variables del modelo, los hombres presentan unas odds aproximadamente 1.94 veces mayores de pertenecer al grupo de ingresos superiores a $50K en comparación con las mujeres. Es impotante que el resultado debe interpretarse como una asociación estadística observada en el conjunto de datos y no como una relación causal. En síntesis, el análisis de los coeficientes de la regresión logística muestra que todas las variables numéricas presentan una asociación positiva con la probabilidad de obtener ingresos superiores a $50K donde factores relacionados con el nivel educativo, la acumulación de capital y la experiencia asociada con la edad presentan una relación más fuerte con los ingresos altos que la cantidad de horas trabajadas por semana.